In [1]:
########################### xESMF Interpolate Zeta from HYCOM to ROMS ###########################
# The purpose of this script is to use the wonderful xesmf package to interpolate surface elevation
# from HYCOM data to the ROMS grid. This script then saves these values to the bry and clm files.

###### IMPORTANT NOTES ######
# - Remember that HYCOM lat/lon indices are not the same as ROMS grid lat/lon indices
#   - Important to keep in mind when comparing results
# - The model2roms workflow is to do the xesmf regridding then run the fill.f90
#   to get rid of any nans
# - HYCOM surface elevation has negative and positive values so we are assuming negative
#   values mean depression and positive values mean elevation 

##############################

In [2]:
# Load in packages
#%matplotlib widget # widget not currently working but instead prevents plots from showing
import matplotlib.pyplot as plt
#import ipywidgets as widgets
import numpy as np
import xarray as xr
import xesmf as xe
import ESMF
import math
from netCDF4 import Dataset
from datetime import datetime, timedelta
from cftime import num2date, date2num
import pandas as pd 

In [3]:
# Load in the HYCOM zeta data
# zeta
hycom_zeta = xr.open_dataset('/scratch/alpine/brun1463/ROMS_scratch/Kakak3_Alpine_2020_scratch/Final_bryclm_conds/Input_data/HYCOM_2020_surf_el_nogaps.nc4') # UPDATE PATH

In [4]:
# Cut out just the time period we care about for the model - the open water season
# zeta
#hycom_zeta_all = hycom_zeta.sel(time=slice('2020-07-01','2020-11-01 21:00:00'))
hycom_zeta_all = hycom_zeta.sel(time=slice('2020-07-01 09:00:00', '2020-11-02 09:00:00')) # this is 2019-07-01 01:00:00 - 2019-11-02 01:00:00 in AKDT
#print('hycom time length: ', len(hycom_zeta_all.time.values))
#print(hycom_zeta_all.time[-6:-1].values)
#input('press enter to continue...')

In [5]:
# Grid's lat/lon is in different convention than HYCOM lat/lon 
# need to make HYCOM match grid's lat lon convention 
hycom_zeta_all['lon_180'] = -(360 - hycom_zeta_all.lon.values)

In [6]:
# Load in the ROMS grid vertical coordinates
#grid = xr.open_dataset('/projects/brun1463/ROMS/Kakak3/Include/KakAKgrd_shelf_big010.nc')  #UPDATE PATH
grid = xr.open_dataset('/projects/brun1463/ROMS/Kakak3_Alpine_2020/Include/KakAKgrd_shelf_big010_smooth006.nc')  #UPDATE PATH

In [7]:
# pull out the angle to rotate the currents to match the grid's u,v
phi = grid.angle[0,0].values # radians 

# Read in the dimensions
time_len = len(hycom_zeta_all.time) # 984?
eta_rho_len = len(grid.eta_rho) # 206
xi_rho_len = len(grid.xi_rho) # 608

# Define other dimension lengths
# eta rho
Mp = len(grid.eta_rho)

# xi rho
Lp = len(grid.xi_rho)

# for u/v points 
Lm, Mm = (Lp-2), (Mp-2) #number/dimension of cells
L,  M  = Lm+1, Mm+1 #number/dimension of psi points

# latitude
lat_len = len(grid.lat_rho)

# longitude
lon_len = len(grid.lon_rho)

# time
# --- Below keeps things in UTC but we want AKDT--- 
# time_tmp_len = np.copy(time_len)
# time2_tmp_len = len(hycom_zeta_all.time)

# # make python do time to seconds 
# # make the hycom times into datetime
# datetime1 = pd.to_datetime(hycom_zeta_all.time.values)

# # convert all the times to seconds since 2000-01-01 (really 1999-12-31 so it start at beginning of year=hour 0)
# time_tmp = ((datetime1[:] - datetime(1999,12,31)).total_seconds() - 86400)
# #print(time_tmp[0:5])
# #print(len(time_tmp))
# #input('press enter to continue...')

# # HYCOM time 
# time_tmp2 = hycom_zeta_all.time.values
# ------

# --- AKDT version --- 
# Make a datetime array to use
time_akdt = np.arange(datetime(2020,7,1,hour=1,minute=0,second=0), datetime(2020,11,2,hour=4,minute=0, second=0),timedelta(hours=3))
time_akdt_dt = pd.to_datetime(time_akdt)
time_tmp_len = len(time_akdt_dt)
datetime1 = time_akdt_dt

# Convert all time to seconds since 2000-01-01 (really 1999-12-31 so it starts at beginning
# of year=hour 0)
time_tmp = ((datetime1[:] - datetime(1999,12,31)).total_seconds() - 86400)

# HYCOM time 
time_tmp2 = hycom_zeta_all.time.values
time2_tmp_len = len(hycom_zeta_all.time)
# ------

# name the variables to fill the netcdf 
# latitude
lat_tmp = grid.lat_rho.values

# longitude
lon_tmp = grid.lon_rho.values

# HYCOM number of lats
hycom_lat_len = len(hycom_zeta_all.lat.values)
#print('hycom_lat_len', hycom_lat_len)

# HYCOM number of lons
hycom_lon_len = len(hycom_zeta_all.lon.values)
#print('hycom_lon_len', hycom_lon_len)

In [8]:
# time_tmp
# print(datetime1[0])
# print(time_tmp2[0])
# print(datetime1[-1])
# print(time_tmp2[-1])
# print(time_tmp_len)
# print(time2_tmp_len)

In [9]:
# Set up the netcdf for the climatology 
# ------------------------------- Create the netCDF file ---------------------------

#name of file I am writing to
zeta_clm = '/scratch/alpine/brun1463/ROMS_scratch/Kakak3_Alpine_2020_scratch/Final_bryclm_conds/zeta_clm_2020_004.nc'   #UPDATE PATH

#create file to write to
nc1 = Dataset(zeta_clm, 'w', format='NETCDF4')

#Global attributes
global_defaults = dict(gridname = 'KakAKgrd_shelf_big010_smooth006.nc',
                      type = 'ROMS grid interpolated HYCOM zeta climatology',
                      history = 'Created by Brianna Undzis',
                      Conventions = 'CF',
                      Institution = 'University of Colorado Boulder',
                      date = str(datetime.today()))
    
#create dictionary for model
d = {}
d = global_defaults

for att, value in d.items():
    setattr(nc1, att, value)

In [10]:
# Create dimensions
nc1.createDimension('xi_rho',  Lp)   # RHO
nc1.createDimension('eta_rho', Mp)
nc1.createDimension('zeta_time', None)
nc1.createDimension('time_HYCOM', None)
nc1.createDimension('one',     1)

<class 'netCDF4._netCDF4.Dimension'>: name = 'one', size = 1

In [11]:
# Create variables # this took several minutes (~9)
# --------------------
# Coordinate Variables
# --------------------
# xi rho
xi_rho = nc1.createVariable('xi_rho', 'd', ('xi_rho',), zlib=True)
xi_rho.long_name = 'xi coordinate of RHO-points'
xi_rho.standard_name = 'projection_xi_coordinate'
xi_rho.units = 'meter'
xi_rho_tmp = np.arange(0, Lp)
xi_rho[:] = xi_rho_tmp[:]

# eta rho
eta_rho = nc1.createVariable('eta_rho', 'd', ('eta_rho',), zlib=True)
eta_rho.long_name = 'eta coordinate of RHO-points'
eta_rho.standard_name = 'projection_eta_coordinate'
eta_rho.units = 'meter'
eta_rho_tmp = np.arange(0, Mp)
eta_rho[:] = eta_rho_tmp[:]

# zeta_time (in seconds)
zeta_time_g = nc1.createVariable('zeta_time', None, ('zeta_time'), zlib=True)
zeta_time_g.long_name = 'seconds since 2000-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
zeta_time_g.units = 'second'
zeta_time_g.field = 'time, scalar, series'
zeta_time_g[:] = time_tmp[:]
    
# time (HYCOM version)
time_g2 = nc1.createVariable('time_HYCOM', None, ('time_HYCOM'), zlib=True)
time_g2.long_name = '3-hour time steps' 
time_g2.units = 'datetime'
time_g2.field = 'time_HYCOM, scalar, series'
time_g2[:] = time_tmp2[:]

# --------------------
# Surface Elevation 
# --------------------

# zeta
zeta_interp_g = nc1.createVariable('zeta', 'f8', ('zeta_time', 'eta_rho', 'xi_rho'), zlib=True)
zeta_interp_g.long_name = 'surface elevation'
zeta_interp_g.units = 'meter'

In [12]:
# Set up the netcdf for the boundary 
# ------------------------------- Create the netCDF file ---------------------------

#name of file I am writing to
zeta_bry = '/scratch/alpine/brun1463/ROMS_scratch/Kakak3_Alpine_2020_scratch/Final_bryclm_conds/zeta_bry_2020_004.nc'  #UPDATE PATH

#create file to write to
nc2 = Dataset(zeta_bry, 'w', format='NETCDF4')

#Global attributes
global_defaults = dict(gridname = 'KakAKgrd_shelf_big010_smooth006.nc',
                      type = 'ROMS grid interpolated HYCOM zeta boundary',
                      history = 'Created by Brianna Undzis',
                      Conventions = 'CF',
                      Institution = 'University of Colorado Boulder',
                      date = str(datetime.today()))
    
#create dictionary for model
d = {}
d = global_defaults

for att, value in d.items():
    setattr(nc2, att, value)

In [13]:
# Create dimensions
nc2.createDimension('xi_rho',  Lp)   # RHO
nc2.createDimension('eta_rho', Mp)
nc2.createDimension('zeta_time', None)
nc2.createDimension('time_HYCOM', None)
nc2.createDimension('one',     1)

<class 'netCDF4._netCDF4.Dimension'>: name = 'one', size = 1

In [14]:
# Create variables
# --------------------
# Coordinate Variables
# --------------------
# xi rho
xi_rho = nc2.createVariable('xi_rho', 'd', ('xi_rho',), zlib=True)
xi_rho.long_name = 'xi coordinate of RHO-points'
xi_rho.standard_name = 'projection_xi_coordinate'
xi_rho.units = 'meter'
xi_rho_tmp = np.arange(0, Lp)
xi_rho[:] = xi_rho_tmp[:]

# eta rho
eta_rho = nc2.createVariable('eta_rho', 'd', ('eta_rho',), zlib=True)
eta_rho.long_name = 'eta coordinate of RHO-points'
eta_rho.standard_name = 'projection_eta_coordinate'
eta_rho.units = 'meter'
eta_rho_tmp = np.arange(0, Mp)
eta_rho[:] = eta_rho_tmp[:]

# zeta_time (in seconds)
zeta_time_g = nc2.createVariable('zeta_time', None, ('zeta_time'), zlib=True)
zeta_time_g.long_name = 'seconds since 2000-01-01 00:00:00' #with initialization of 2000-01-01 00:00:00
zeta_time_g.units = 'second'
zeta_time_g.field = 'time, scalar, series'
zeta_time_g[:] = time_tmp[:]
    
# time (HYCOM version)
time_g2 = nc2.createVariable('time_HYCOM', None, ('time_HYCOM'), zlib=True)
time_g2.long_name = '3-hour time steps' 
time_g2.units = 'datetime'
time_g2.field = 'time_HYCOM, scalar, series'
time_g2[:] = time_tmp2[:]

# ---------------------------
# Boundary Surface Elevation
# ---------------------------

# zeta
# zeta_west
zeta_west_g = nc2.createVariable('zeta_west', 'f8', ('zeta_time', 'eta_rho'), zlib=True)
zeta_west_g.long_name = 'surface elevation at western boundary'
zeta_west_g.units = 'meter'

# zeta_north
zeta_north_g = nc2.createVariable('zeta_north', 'f8', ('zeta_time', 'xi_rho'), zlib=True)
zeta_north_g.long_name = 'surface elevation at northern boundary'
zeta_north_g.units = 'meter'

# zeta_east
zeta_east_g = nc2.createVariable('zeta_east', 'f8', ('zeta_time', 'eta_rho'), zlib=True)
zeta_east_g.long_name = 'surface elevation at eastern boundary'
zeta_east_g.units = 'meter'

In [15]:
# Set the input and output grids, and sepcify the lat/lon
# Since we are looking at zeta for now, we will use lon_rho and lat_rho as the primary lat/lon for the grid 
# Input grid (HYCOM)
ds_in_hycom = hycom_zeta_all.copy() # need to use lon_180 for this grid 
ds_in_hycom['lon_360'] = ds_in_hycom.lon.values
ds_in_hycom['lon'] = ds_in_hycom.lon_180.values

# Output grid (ROMS rho grid)
#ds_out_rho = grid_vertical
ds_out_rho = grid.copy()
ds_out_rho['lat'] = (('eta_rho', 'xi_rho'), ds_out_rho.lat_rho.values)
ds_out_rho['lon'] = (('eta_rho', 'xi_rho'), ds_out_rho.lon_rho.values)

# Add masks 
# ex: ds["mask"] = xr.where(~np.isnan(ds["zeta"].isel(ocean_time=0)), 1, 0)
# Input grid (HYCOM)
# this is only a surface mask - which is what we want 
ds_in_hycom_mask = xr.where(~np.isnan(ds_in_hycom['surf_el'][0,:,:].values), 1, 0) 
ds_in_hycom['mask'] = (('lat', 'lon'), ds_in_hycom_mask)

# Output grid (ROMS rho grid)
ds_out_rho['mask'] = (('eta_rho', 'xi_rho'), ds_out_rho.mask_rho.values)

In [16]:
# Regrid from HYCOM grid to rho grid with the masks included and extrapolation used 
regridder_hycom2rho = xe.Regridder(ds_in_hycom, ds_out_rho, method="bilinear", extrap_method='nearest_s2d') #extrap_method="nearest_s2d"
regridder_hycom2rho

xESMF Regridder 
Regridding algorithm:       bilinear 
Weight filename:            bilinear_126x238_206x608.nc 
Reuse pre-computed weights? False 
Input grid shape:           (126, 238) 
Output grid shape:          (206, 608) 
Periodic in longitude?      False

In [17]:
# Save the weights - only need to do this once
fn_hycom2rho = regridder_hycom2rho.to_netcdf('regrid_hycom2rho_weights.nc')
#print(fn_hycom2u)

In [18]:
# Now use the regridder/weights to regrid zeta  
dr_hycom2rho_zeta = hycom_zeta_all['surf_el'].copy()
dr_out_hycom2rho_zeta = regridder_hycom2rho(dr_hycom2rho_zeta) 
dr_out_hycom2rho_zeta

<xarray.DataArray (time: 993, eta_rho: 206, xi_rho: 608)>
array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.31151593, -0.31250557, -0.31349513, ..., -0.28137958,
         -0.28137976, -0.28138   ],
        [-0.31211406, -0.31310403, -0.3140939 , ..., -0.28068528,
         -0.2806854 , -0.2806856 ],
        [-0.31271228, -0.31370252, -0.3146927 , ..., -0.27999094,
         -0.279991  , -0.27999115]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
...
        [-0.3156562 , -0.3174014 , -0.3191465 , ..., -0.2729651 ,
         -0.27306154, -0.27315807],
        [-0.31657016, -0.31831592, -0.32006156, ..., -0.27201635,
         -0.27211273, -0.2722092 ],
        [-0.31748423, -0.31923053, -0.32097667, ..., -0.27106753,
         -0.27116388, -0.2712603 ]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.3058486 , -0.30773368, -0.3096256 , ..., -0.2789651 ,
         -0.27906156, -0.2791581 ],
        [-0.30671805, -0.30856928, -0.31042734, ..., -0.27801633,
         -0.27811274, -0.2782092 ],
        [-0.30758312, -0.30940047, -0.31122467, ..., -0.2770675 ,
         -0.27716386, -0.2772603 ]]], dtype=float32)
Coordinates:
  * time          (time) datetime64[ns] 2020-07-01T09:00:00 ... 2020-11-02T09...
  * xi_rho        (xi_rho) float64 0.0 1.0 2.0 3.0 ... 604.0 605.0 606.0 607.0
  * eta_rho       (eta_rho) float64 0.0 1.0 2.0 3.0 ... 202.0 203.0 204.0 205.0
    lat_rho       (eta_rho, xi_rho) float64 70.38 70.38 70.38 ... 70.72 70.71
    lon_rho       (eta_rho, xi_rho) float64 -153.7 -153.6 ... -141.2 -141.1
    grid_mapping  int32 ...
Attributes:
    regrid_method:  bilinear

In [19]:
# Use fill.f90 to fill the nans in the array
# Import fill.f90 from model2roms to see how to 
# use this/if it can be used to get rid of nans 
from numpy import f2py
with open('/projects/brun1463/ROMS/Kakak3_Alpine_2020/Scripts/Bryclm_conds/fill.f90') as sourcefile2:
    sourcecode2 = sourcefile2.read()
f2py.compile(sourcecode2, modulename='fill', extension='.f90')
import fill

running build
running config_cc
unifing config_cc, config, build_clib, build_ext, build commands --compiler options
running config_fc
unifing config_fc, config, build_clib, build_ext, build commands --fcompiler options
running build_src
build_src
building extension "fill" sources
f2py options: []
f2py:> /tmp/tmpashpe654/src.linux-x86_64-3.7/fillmodule.c
creating /tmp/tmpashpe654/src.linux-x86_64-3.7
Reading fortran codes...
	Reading file '/tmp/tmp2dogs4h3.f90' (format:free)
Post-processing...
	Block: fill
			Block: extrapolate
				Block: fill
Post-processing (stage 2)...
	Block: fill
		Block: unknown_interface
			Block: extrapolate
				Block: fill
Building modules...
	Building module "fill"...
		Constructing F90 module support for "extrapolate"...
			Constructing wrapper function "extrapolate.fill"...
			  za = fill(i1,i2,j1,j2,tx,critx,cor,mxs,za,[nx,ny,overwrite_za])
	Wrote C/API module "fill" to file "/tmp/tmpashpe654/src.linux-x86_64-3.7/fillmodule.c"
	Fortran 90 wrappers are saved

In [20]:
# Define a function to call to do the filling, taken from model2roms
def laplacefilter(field, threshold, toxi, toeta):
    undef = 2.0e+35 
    tx = 0.9 * undef
    critx = 0.01
    cor = 1.6
    mxs = 10

    field = np.where(abs(field) > threshold, undef, field)

    field = fill.extrapolate.fill(int(1), int(toxi),
                                int(1), int(toeta),
                                float(tx), float(critx), float(cor), float(mxs),
                                np.asarray(field, order='F'),
                                int(toxi),
                                int(toeta))
    return field

In [21]:
# Loop through time to fill in the nans
# Make some variables first
toxi = xi_rho_len
toeta = eta_rho_len

# Make an array to hold the new data without nans
#print('got here 1')
dr_out_hycom2rho_zeta_nonan = np.empty((time_len, eta_rho_len, xi_rho_len))
#print('got here 2')

# Make a copy of the OG array to work with
#print('got here 3')
dr_out_hycom2rho_zeta_cp1 = dr_out_hycom2rho_zeta.copy() 
#print('got here 4')

# Loop through depth to replace all the nans with real values 
# Loop through time
for t in range(time_len):
    # Print the time 
    print('t: ', t)

    # Pull out the horizontal 'field' for that time
    field = dr_out_hycom2rho_zeta_cp1[t,:,:]

    # Use the Laplace Filter to get rid of nans
    field = laplacefilter(field, 1000, toxi, toeta)

    # Multiply by the rho mask 
    field = field * ds_out_rho.mask.values

    # Check to see if there are any nans
    #print('nans: ', np.where(np.isnan(field)))
    #print('nanmin: ', np.nanmin(field))
    #print('nanmax: ', np.nanmax(field))
    #input('press enter to continue...')

    # Save this field to a new array
    dr_out_hycom2rho_zeta_nonan[t,:,:] = field

t:  0
t:  1
t:  2
t:  3
t:  4
t:  5
t:  6
t:  7
t:  8
t:  9
t:  10
t:  11
t:  12
t:  13
t:  14
t:  15
t:  16
t:  17
t:  18
t:  19
t:  20
t:  21
t:  22
t:  23
t:  24
t:  25
t:  26
t:  27
t:  28
t:  29
t:  30
t:  31
t:  32
t:  33
t:  34
t:  35
t:  36
t:  37
t:  38
t:  39
t:  40
t:  41
t:  42
t:  43
t:  44
t:  45
t:  46
t:  47
t:  48
t:  49
t:  50
t:  51
t:  52
t:  53
t:  54
t:  55
t:  56
t:  57
t:  58
t:  59
t:  60
t:  61
t:  62
t:  63
t:  64
t:  65
t:  66
t:  67
t:  68
t:  69
t:  70
t:  71
t:  72
t:  73
t:  74
t:  75
t:  76
t:  77
t:  78
t:  79
t:  80
t:  81
t:  82
t:  83
t:  84
t:  85
t:  86
t:  87
t:  88
t:  89
t:  90
t:  91
t:  92
t:  93
t:  94
t:  95
t:  96
t:  97
t:  98
t:  99
t:  100
t:  101
t:  102
t:  103
t:  104
t:  105
t:  106
t:  107
t:  108
t:  109
t:  110
t:  111
t:  112
t:  113
t:  114
t:  115
t:  116
t:  117
t:  118
t:  119
t:  120
t:  121
t:  122
t:  123
t:  124
t:  125
t:  126
t:  127
t:  128
t:  129
t:  130
t:  131
t:  132
t:  133
t:  134
t:  135
t:  136
t:  137
t:  13

In [22]:
# Save the interpolated/regridded zeta to the netcdf files
# to the climatology file
zeta_interp_g[:,:,:] = dr_out_hycom2rho_zeta_nonan[:,:,:]

# to the boundary file
zeta_west_g[:,:] = dr_out_hycom2rho_zeta_nonan[:,:,0]
zeta_north_g[:,:] = dr_out_hycom2rho_zeta_nonan[:,-1,:]
zeta_east_g[:,:] = dr_out_hycom2rho_zeta_nonan[:,:,-1]

In [23]:
# Close the netcdfs
nc1.close()
nc2.close()